# Baseline Churn Model Check + Initial Behavioral Drift Score

Input: `data/features_and_label.parquet` (customer-month rows from notebook 03, label `is_churn`)

Output: drift features + `drift_score` per customer-month

1. Verify that the baseline churn models from 06 beat trivial baselines and are not fooled by leakage
2. Build the first behavioral-drift score and test whether it carries churn signal beyond th existing features


## 1. Setup + Load Data

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [3]:
DATA_PATH     = Path('../data/features_and_label.parquet')
OUT_PATH      = Path('../data/drift_scores.parquet')
TARGET        = 'is_churn'

TEST_FRACTION = 0.20   # same as notebook 06: latest 20% of distinct months are the test set

PURGE_MONTHS  = 3      # = the 90-day label window (~3 months)

K_BASELINE    = 3      # a customer's "normal" = mean of their previous 3 ACTIVE months
CLIP_Z        = 3.0    # cap each standardized drift component at +/-3 so one freak month can't dominate
RANDOM_STATE  = 42

In [4]:
df = pd.read_parquet(DATA_PATH)

# CustomerMonth is stored as a Period[M], so we'll convert to a month-start timestamp (same handling as notebook 06)
cm = df['CustomerMonth']
df['CustomerMonth'] = cm.dt.to_timestamp() if isinstance(cm.dtype, pd.PeriodDtype) else pd.to_datetime(cm)

df = df.dropna(subset=[TARGET, 'CustomerMonth']).copy()
df[TARGET] = df[TARGET].astype(int)
df['is_first_month'] = df['is_first_month'].astype(int)   # bool -> 0/1 so every model treats it as a plain number

# sort by customer then month
# all "previous months" logic below (shift/rolling) depends on this order
df = df.sort_values(['Customer ID', 'CustomerMonth']).reset_index(drop=True)

print(f'Rows: {len(df):,} | customers: {df["Customer ID"].nunique():,} | churn rate: {df[TARGET].mean():.2%}')
print(f'Months: {df.CustomerMonth.min():%Y-%m} to {df.CustomerMonth.max():%Y-%m}')
display(df.head())

Rows: 20,880 | customers: 5,248 | churn rate: 38.99%
Months: 2009-12 to 2011-09


,Customer ID,CustomerMonth,purchase_frequency,total_spend,avg_order_value,unique_items,days_since_last_purchase,is_first_month,is_churn
0,12346,2010-03-01,1,27.05,27.05,5,NaN,1,1
1,12346,2010-06-01,1,142.31,142.31,19,118.0,0,1
2,12346,2011-01-01,1,149.76,149.76,1,203.0,0,1
3,12347,2010-10-01,1,611.53,611.53,40,NaN,1,0
4,12347,2010-12-01,1,711.79,711.79,31,37.0,0,0


## 2. Chronological Split with a Purge Gap

Same idea as notebook 06: train on earlier months & test on the latest 20% of distinct months. A random split would be wrong here, because it lets the model learn from the future.

- Added a purge gap of 3 months at the end of train

A training row from month *M* is labeled using purchases up to ~90 days after its last purchase, which reaches into months *M+1 ... M+3*. 

If the test period starts right after train, the final training labels are partly built from purchases that fall *inside the test period*, so the model has indirectly seen test-period behavior. 

Dropping the last `PURGE_MONTHS` months of train removes that overlap.

The cost is a smaller training set; the benefit is a test score that reflects what you'd get in production.


In [5]:
def make_split(data, purge_months):
    months = np.sort(data['CustomerMonth'].unique())
    cutoff_idx = min(max(1, int(np.floor(len(months) * (1 - TEST_FRACTION)))), len(months) - 1)
    cutoff = pd.Timestamp(months[cutoff_idx])
    train_end = cutoff - pd.DateOffset(months=purge_months)   # train uses months strictly before this
    train = data[data['CustomerMonth'] < train_end]
    test  = data[data['CustomerMonth'] >= cutoff]
    return train, test, cutoff, train_end

train_df, test_df, cutoff, train_end = make_split(df, PURGE_MONTHS)
train_nopurge, _, _, _ = make_split(df, 0)

print(f'Test starts:               {cutoff:%Y-%m}')
print(f'Purged train ends before:  {train_end:%Y-%m}  -> {len(train_df):,} rows, churn {train_df[TARGET].mean():.2%}')
print(f'Un-purged train (as in 06): {len(train_nopurge):,} rows')
print(f'Test:                      {len(test_df):,} rows, churn {test_df[TARGET].mean():.2%}')
assert train_df['CustomerMonth'].max() < test_df['CustomerMonth'].min(), 'train must precede test'

Test starts:               2011-05
Purged train ends before:  2011-02  -> 14,103 rows, churn 40.19%
Un-purged train (as in 06): 16,686 rows
Test:                      4,194 rows, churn 35.24%


## 3. Checking the Baseline Models Work

Using same three models and hyperparameters as notebook 06 (logistic regression, random forest, XGBoost), so results are comparable.

Headline metrics are ROC AUC, PR AUC, and lift in the top 10%

All three are threshold-free.

The models use `class_weight='balanced'` / `scale_pos_weight`, which shifts predicted probabilities so that the default 0.5 cutoff is arbitrary, and precision/recall/F1 at 0.5 mostly reflect that choice rather than model quality. 

Lift at 10% answers the business question "if we contact the 10% of customers the model ranks riskiest, how many more churners do we catch than by picking at random?"

In [6]:
BASE_FEATURES = ['purchase_frequency', 'total_spend', 'avg_order_value',
                 'unique_items', 'days_since_last_purchase', 'is_first_month']

def lift_at(y, score, frac=0.10):
    # churn rate among the top `frac` highest-scored rows, divided by the overall churn rate
    # Ties (e.g. a constant dummy score, or the many neutral-0 drift rows) are broken randomly with a fixed seed
    # Without this, tied rows would be ranked by row order and a constant score could show a fake lift
    y = np.asarray(y); score = np.asarray(score); k = max(1, int(len(y) * frac))
    order = np.random.default_rng(RANDOM_STATE).permutation(len(y))
    top = order[np.argsort(-score[order], kind='stable')[:k]]
    return y[top].mean() / y.mean()

def score_metrics(y, s):
    return {'ROC AUC': roc_auc_score(y, s), 'PR AUC': average_precision_score(y, s), 'Lift@10%': lift_at(y, s)}

def build_models(y_train):
    # same hyperparameters from notebook 06 on purpose (comparability)
    # Median imputation is only
    # needed for days_since_last_purchase in first-month rows; 
    # is_first_month tells the model which rows were imputed.
    return {
        'Logistic regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
        'Random forest': make_pipeline(SimpleImputer(strategy='median'),
            RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        'XGBoost': make_pipeline(SimpleImputer(strategy='median'),
            XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
                          eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)),
    }

def fit_and_eval(tr, te, features):
    models = build_models(tr[TARGET])
    rows, fitted = {}, {}
    for name, m in models.items():
        m.fit(tr[features], tr[TARGET])
        p_te = m.predict_proba(te[features])[:, 1]
        p_tr = m.predict_proba(tr[features])[:, 1]
        rows[name] = {**score_metrics(te[TARGET], p_te), 'Train ROC AUC': roc_auc_score(tr[TARGET], p_tr)}
        fitted[name] = m
    return pd.DataFrame(rows).T, fitted


## results
results_purged, fitted_base = fit_and_eval(train_df, test_df, BASE_FEATURES)
print(f'Test churn rate (= PR AUC of a random model): {test_df[TARGET].mean():.3f}')
display(results_purged.round(3))

Test churn rate (= PR AUC of a random model): 0.352


,ROC AUC,PR AUC,Lift@10%,Train ROC AUC
Logistic regression,0.766,0.626,2.066,0.734
Random forest,0.713,0.533,1.734,1.000
XGBoost,0.765,0.604,2.025,0.804


### Sanity Checks

1. **Trivial baselines** A model that predicts the class prior should score ROC AUC ~0.5 and PR AUC ~ churn rate. A single raw column (recency) is the "simple rule" every real model must beat
2. **Shuffled-label test** Train on randomly permuted labels and score on the real test labels; the *average* AUC over many shuffles should be ~0.5. This is a coarse plumbing check (it would catch, for example, test rows accidentally ending up in train). It cannot detect leakage inside the features, which is what the purge and the drift truncation test below address
3. **Train-vs-test gap** A huge gap (for example train AUC ~1.0) means overfitting
4. **Purge effect** Compare the unpurged split of notebook 06 against the purged split
5. **Stability by month** One good test month can hide a bad one

### 1. Trivial Baselines
The real models should clearly beat the dummy and the recency rule on ROC AUC and PR AUC, and have Lift@10% above 1

In [7]:
dummy = DummyClassifier(strategy='prior').fit(train_df[BASE_FEATURES], train_df[TARGET])
dummy_s = dummy.predict_proba(test_df[BASE_FEATURES])[:, 1]

# single-feature rule: raw days_since_last_purchase (first-month NaN -> train median)
# not sign-flipped, so an AUC below 0.5 would mean longer gaps actually go with LESS churn here
rec = test_df['days_since_last_purchase'].fillna(train_df['days_since_last_purchase'].median())

dummy_metrics = score_metrics(test_df[TARGET], dummy_s)
# a constant score ranks nobody, so its Lift@10% is a pure coin-flip
# One random draw is noisy (+/-~13% at this test size), 
# so report the AVERAGE over many random rankings instead; it should land at ~1.0 by construction
rs = np.random.default_rng(RANDOM_STATE)
dummy_metrics['Lift@10%'] = np.mean([lift_at(test_df[TARGET], rs.random(len(test_df))) for _ in range(300)])

baselines = pd.DataFrame({
    'Dummy (class prior)':       dummy_metrics,
    'Rule: recency only':        score_metrics(test_df[TARGET], rec),
}).T
display(pd.concat([baselines, results_purged[['ROC AUC', 'PR AUC', 'Lift@10%']]]).round(3))


,ROC AUC,PR AUC,Lift@10%
Dummy (class prior),0.500,0.352,1.001
Rule: recency only,0.715,0.582,1.991
Logistic regression,0.766,0.626,2.066
Random forest,0.713,0.533,1.734
XGBoost,0.765,0.604,2.025


### 2. Shuffled-Label Test
A shuffled-label *mean* AUC far from 0.5 (say outside 0.45-0.55) means a plumbing bug

Individual shuffles can land well away from 0.5; that is expected

In [8]:
# each run trains on pure-noise labels and is scored on the REAL test labels
# single runs vary a lot (a noise-trained model has random weights over features that are individually predictive, 
# so its AUC lands above OR below 0.5)
# so judge the MEAN over many shuffles, not any one run
rng = np.random.default_rng(RANDOM_STATE)
shuf = []
for _ in range(20):
    y_shuf = pd.Series(rng.permutation(train_df[TARGET].values), index=train_df.index)
    m = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                      LogisticRegression(max_iter=2000, class_weight='balanced'))
    m.fit(train_df[BASE_FEATURES], y_shuf)
    shuf.append(roc_auc_score(test_df[TARGET], m.predict_proba(test_df[BASE_FEATURES])[:, 1]))
print(f'Shuffled-label ROC AUC over {len(shuf)} shuffles: mean {np.mean(shuf):.3f}, std {np.std(shuf):.3f}, range {min(shuf):.3f} to {max(shuf):.3f}   (expect mean ~0.5)')


Shuffled-label ROC AUC over 20 shuffles: mean 0.536, std 0.152, range 0.265 to 0.737   (expect mean ~0.5)


### 3. Train-vs-Test Gap
A very large Train-minus-Test gap (random forest with unlimited depth usually shows ~1.0 train AUC) means the model is memorizing. 

In [9]:
gap = (results_purged['Train ROC AUC'] - results_purged['ROC AUC']).rename('Train - Test AUC')
display(gap.round(3).to_frame())

,Train - Test AUC
Logistic regression,-0.032
Random forest,0.287
XGBoost,0.039


### 4. Effect of the Purge Gap
If the purged AUC is noticeably lower than the un-purged AUC, notebook 06 was slightly optimistic and the purged number is the one to report

* With the un-purged split, some training labels were built from purchases that happen in the test period. The model has indirectly seen a bit of the answer, so its test score is inflated. The model itself isn’t better

In [10]:
res_nopurge, _ = fit_and_eval(train_nopurge, test_df, BASE_FEATURES)
purge_cmp = pd.DataFrame({'ROC AUC (no purge, notebook 06)': res_nopurge['ROC AUC'],
                          'ROC AUC (purged)':             results_purged['ROC AUC']})
purge_cmp['Difference'] = purge_cmp.iloc[:, 0] - purge_cmp.iloc[:, 1]
display(purge_cmp.round(3))

# no purge is yields higher scores

,"ROC AUC (no purge, notebook 06)",ROC AUC (purged),Difference
Logistic regression,0.766,0.766,-0.000
Random forest,0.719,0.713,0.006
XGBoost,0.768,0.765,0.003


### 5. Stability by Test Month
Because `total_spend` and `avg_order_value` are identical on most rows, logistic-regression coefficients for the two are not individually interpretable (they split one signal)

Tree models are unaffected.

In [11]:
per_month = {}
for name in ['Logistic regression', 'XGBoost']:
    p = pd.Series(fitted_base[name].predict_proba(test_df[BASE_FEATURES])[:, 1], index=test_df.index)
    per_month[name] = {
        f'{mth:%Y-%m}': (roc_auc_score(g[TARGET], p.loc[g.index]) if g[TARGET].nunique() == 2 else np.nan)
        for mth, g in test_df.groupby('CustomerMonth')}
display(pd.DataFrame(per_month).T.round(3))


,2011-05,2011-06,2011-07,2011-08,2011-09
Logistic regression,0.791,0.771,0.757,0.750,0.714
XGBoost,0.791,0.771,0.751,0.753,0.719


### Understanding
Positive value: a higher value of that feature goes with more churn

Negative value: a higher value goes with less churn

Size: because everything is on the same scale, we can compare magnitudes across features

The table is sorted from most protective (top) to most churn-linked (bottom)


| Feature | Expected sign | Reasoning |
|---|---|---|
| `purchase_frequency` | negative | buying more often, less churn |
| `total_spend` | negative | bigger spenders are more engaged |
| `unique_items` | negative | broader engagement |
| `days_since_last_purchase` | positive | longer gap, drifting away |
| `is_first_month` | unclear | new customers can go either way |

In [12]:
# logistic regression coefficients: do the signs make business sense? -
coefs = pd.Series(fitted_base['Logistic regression'][-1].coef_[0], index=BASE_FEATURES).sort_values()
display(coefs.round(3).rename('Std. coefficient (+ = more churn)').to_frame())

,Std. coefficient (+ = more churn)
purchase_frequency,-0.526
total_spend,-0.407
unique_items,-0.086
avg_order_value,-0.065
days_since_last_purchase,0.523
is_first_month,0.593


## Behavioral Drift Features

Drift compares each customer-month with that customer's own recent normal

To-Do: 
- Build drift at the customer-month grain
- baseline = mean of the customer's previous `K_BASELINE`=3 *active* months
customer-month rows only exist for months with a purchase, so "previous 3 months" in calendar terms would often be empty

Trade-off: a customer returning after a long silence is compared with their last active months, not with the silent months in between, and that silence is already captured by `days_since_last_purchase`


| Feature | Meaning | "Worse" direction |
|---|---|---|
| `freq_drift` | orders this month vs. normal | negative |
| `spend_drift` | spend this month vs. normal | negative |
| `items_drift` | distinct products vs. normal | negative |
| `aov_drift` | avg order value vs. normal | negative |
| `gap_drift` | days-since-last-purchase vs. the customer's usual gap | **positive** (slower) |
| `n_prior_months` | active months before this one (how much history backs the drift) | N/A |


In [13]:
DRIFT_BASES = { # new feature name : source column in features_and_label
    'freq_drift':  'purchase_frequency',
    'spend_drift': 'total_spend',
    'items_drift': 'unique_items',
    'aov_drift':   'avg_order_value',
}

def trailing_baseline(frame, col, k=K_BASELINE):
    # mean of the customer's previous k rows (active months)
    # shift(1) excludes the current row -> no leakage
    return frame.groupby('Customer ID')[col].transform(lambda s: s.shift(1).rolling(k, min_periods=1).mean())

def compute_drift_raw(frame):
    # returns raw drift columns with NaN kept where no baseline exists 
    # (NaN != 0: it means 'unknown')
    out = pd.DataFrame(index=frame.index)
    for name, col in DRIFT_BASES.items():
        out[name] = np.log1p(frame[col]) - np.log1p(trailing_baseline(frame, col))
    # positive = current gap is longer than the customer's usual gap
    out['gap_drift'] = np.log1p(frame['days_since_last_purchase']) - np.log1p(trailing_baseline(frame, 'days_since_last_purchase'))
    out['n_prior_months'] = frame.groupby('Customer ID').cumcount()
    return out

drift_raw = compute_drift_raw(df)
drift_cols = list(DRIFT_BASES) + ['gap_drift']

print('Share of rows with a computable drift value (the rest are first/second-month rows):')
display(drift_raw[drift_cols].notna().mean().round(3).to_frame('coverage'))
display(drift_raw[drift_cols].describe().round(3))

Share of rows with a computable drift value (the rest are first/second-month rows):


,coverage
freq_drift,0.749
spend_drift,0.749
items_drift,0.749
aov_drift,0.749
gap_drift,0.579


,freq_drift,spend_drift,items_drift,aov_drift,gap_drift
count,15632.000,15632.000,15632.000,15632.000,12099.000
mean,-0.020,-0.133,-0.148,-0.105,-0.075
std,0.270,0.738,0.706,0.658,0.840
min,-1.711,-7.829,-4.538,-5.656,-4.818
25%,-0.154,-0.510,-0.511,-0.431,-0.566
50%,0.000,-0.088,-0.105,-0.059,-0.043
75%,0.000,0.289,0.272,0.264,0.451
max,1.281,4.660,3.314,4.660,3.555


In [15]:
# leakage guard: recompute drift for one customer by hand and compare
# A customer's drift in month M must not change if we delete every later month from the data
cid = df.groupby('Customer ID').size().sort_values().index[-1] # a customer with a long history
sub = df[df['Customer ID'] == cid]
mid = len(sub) // 2
truncated = sub.iloc[:mid + 1] # pretend the data ends at month `mid`
full_val  = drift_raw.loc[sub.index[mid], 'freq_drift']
trunc_val = compute_drift_raw(truncated).loc[truncated.index[mid], 'freq_drift']
print(f'freq_drift at month {mid}: full data {full_val:.6f} | data truncated at that month {trunc_val:.6f}')
assert np.isclose(full_val, trunc_val, equal_nan=True), 'drift uses future information!'
print('OK: drift does not depend on later months')

freq_drift at month 11: full data 0.587787 | data truncated at that month 0.587787
OK: drift does not depend on later months
